# 03.3 — N-gram Language Models

**Problem it solves:** Assign a probability to a sequence of words. Essential for speech recognition, machine translation, and understanding how language works statistically.

**Core idea:** Approximate P(w1, w2, ..., wn) = ∏ P(wi | w1...wi-1) using the Markov assumption: only the last N-1 words matter (N-gram context).

**Perplexity:** The standard metric. Measures how surprised the model is by the test text. Lower is better. Perplexity ≈ effective branching factor at each step.

---

In [ ]:
import math
import re
from collections import Counter, defaultdict
from typing import Generator

def tokenize(text):
    return re.findall(r'\b[a-z\']+\b', text.lower())

class NGramLanguageModel:
    """
    N-gram language model with add-k smoothing.
    Supports unigram, bigram, and trigram models.
    """
    
    BOS = '<s>'   # beginning of sentence
    EOS = '</s>'  # end of sentence
    
    def __init__(self, n: int = 2, k: float = 0.01):
        """
        n: context size (2=bigram, 3=trigram)
        k: add-k smoothing. k=1 = Laplace, k→0 = MLE
        """
        self.n = n
        self.k = k
        self.ngram_counts: Counter = Counter()
        self.context_counts: Counter = Counter()
        self.vocab: set = set()
    
    def _get_ngrams(self, tokens: list[str]) -> Generator:
        # Pad with BOS/EOS markers
        padded = [self.BOS] * (self.n - 1) + tokens + [self.EOS]
        for i in range(len(padded) - self.n + 1):
            yield tuple(padded[i:i + self.n])
    
    def fit(self, sentences: list[str]):
        for sent in sentences:
            tokens = tokenize(sent)
            self.vocab.update(tokens)
            for ngram in self._get_ngrams(tokens):
                self.ngram_counts[ngram] += 1
                self.context_counts[ngram[:-1]] += 1  # prefix
        
        self.vocab.update([self.BOS, self.EOS])
        return self
    
    def log_prob(self, ngram: tuple) -> float:
        """Log P(word | context) with add-k smoothing."""
        context = ngram[:-1]
        count = self.ngram_counts.get(ngram, 0)
        context_count = self.context_counts.get(context, 0)
        V = len(self.vocab)
        # Add-k smoothing: P(w|context) = (count + k) / (context_count + k*V)
        prob = (count + self.k) / (context_count + self.k * V)
        return math.log(prob)
    
    def sentence_log_prob(self, sentence: str) -> float:
        tokens = tokenize(sentence)
        log_p = 0.0
        for ngram in self._get_ngrams(tokens):
            log_p += self.log_prob(ngram)
        return log_p
    
    def perplexity(self, sentences: list[str]) -> float:
        """
        Perplexity = exp(-1/N * sum(log P(wi|context)))
        = geometric mean of inverse word probabilities
        Lower = model assigns higher probability to the text = better.
        """
        total_log_prob = 0.0
        total_tokens = 0
        for sent in sentences:
            tokens = tokenize(sent)
            total_log_prob += self.sentence_log_prob(sent)
            total_tokens += len(tokens) + 1  # +1 for EOS
        
        avg_log_prob = total_log_prob / total_tokens
        return math.exp(-avg_log_prob)
    
    def generate(self, seed: str = '', max_tokens: int = 20) -> str:
        """Sample from the language model."""
        import random
        
        seed_tokens = tokenize(seed) if seed else []
        context = tuple([self.BOS] * (self.n - 1) + seed_tokens)
        context = context[-(self.n-1):] if self.n > 1 else ()
        
        generated = list(seed_tokens)
        
        for _ in range(max_tokens):
            # Get all possible next words given context
            candidates = [
                ngram[-1] for ngram in self.ngram_counts
                if ngram[:-1] == context and ngram[-1] != self.EOS
            ]
            if not candidates:
                break
            
            # Sample proportional to counts
            counts = [self.ngram_counts[context + (w,)] for w in candidates]
            total = sum(counts)
            probs = [c/total for c in counts]
            
            next_word = random.choices(candidates, weights=probs)[0]
            if next_word == self.EOS:
                break
            
            generated.append(next_word)
            context = (context + (next_word,))[-(self.n-1):] if self.n > 1 else ()
        
        return ' '.join(generated)

# Train on Shakespeare-like text
train_sentences = [
    "the quick brown fox jumps over the lazy dog",
    "the dog barked loudly at the fox",
    "a quick brown rabbit jumped over the hill",
    "the fox ran quickly away from the dog",
    "the lazy dog slept under the big tree",
    "a brown fox and a brown dog played together",
    "the dog chased the fox through the forest",
    "quick brown animals run through the green fields",
    "the fox hid quietly behind the old tree",
    "dogs and foxes rarely play together in the wild",
]

bigram_lm  = NGramLanguageModel(n=2, k=0.01).fit(train_sentences)
trigram_lm = NGramLanguageModel(n=3, k=0.01).fit(train_sentences)

print(f"Vocab size: {len(bigram_lm.vocab)}")
print(f"Bigram types: {len(bigram_lm.ngram_counts)}")
print(f"Trigram types: {len(trigram_lm.ngram_counts)}")

In [ ]:
# Perplexity: in-domain vs out-of-domain
test_in_domain = [
    "the fox ran over the hill",
    "a brown dog jumped quickly",
]
test_out_domain = [
    "machine learning is a powerful technology",
    "the neural network achieved high accuracy",
]

for lm_name, lm in [('Bigram', bigram_lm), ('Trigram', trigram_lm)]:
    ppl_in  = lm.perplexity(test_in_domain)
    ppl_out = lm.perplexity(test_out_domain)
    print(f"{lm_name}:  in-domain PPL={ppl_in:.1f}  out-of-domain PPL={ppl_out:.1f}")

print()
print("In-domain text should have lower perplexity.")
print("Trigram should be lower on in-domain (more context = better fit)")
print("but may be higher out-of-domain (more overfit to training distribution)")

In [ ]:
# Generation
import random
random.seed(42)

print("Generated text (bigram model):")
for _ in range(5):
    print(f"  {bigram_lm.generate('the', max_tokens=10)}")

print("\nGenerated text (trigram model):")
for _ in range(5):
    print(f"  {trigram_lm.generate('the quick', max_tokens=10)}")

In [ ]:
# Smoothing comparison: how k affects perplexity
print("Effect of smoothing parameter k on test perplexity:")
for k in [0, 0.001, 0.01, 0.1, 1.0]:
    try:
        lm = NGramLanguageModel(n=2, k=k).fit(train_sentences)
        ppl = lm.perplexity(test_in_domain)
        print(f"  k={k:.3f}  PPL={ppl:.2f}")
    except (ValueError, OverflowError):
        print(f"  k={k:.3f}  ERROR (zero probability)")

print()
print("Too little smoothing: zero probs -> infinite perplexity")
print("Too much smoothing: uniform-ish distribution -> high perplexity")
print("Sweet spot: small k, or use Kneser-Ney (better but more complex)")

## Summary

**What n-gram LMs teach you:**
- Language is non-Markovian: dependencies span sentences, not just 2-3 words
- Smoothing is essential — unseen n-grams are common even in large corpora
- Perplexity is a proxy for compression — lower PPL = better compression of the test set

**Why neural LMs replaced them:**
- N-gram models can't generalize: 'cat sat on mat' tells nothing about 'dog lay on rug'
- Memory: storing all trigrams for 1B tokens requires massive RAM
- Neural models learn shared representations across similar contexts

**Still relevant:** LM perplexity is still used to evaluate neural models. Kneser-Ney smoothing is still used in some production systems. N-gram LMs are faster at inference.

---
**Next:** 03.4 — Edit Distance & Spell Correction